In [ ]:
import  cart_model
from tqdm.auto import tqdm

import random
random.seed(0)
context = cart_model.Cart_model(human=False)

In [ ]:

from itertools import product

sample_parameters = cart_model.ActorPolicyContinuousSpace.get_paramters_samples(context)
keys = list(sample_parameters.keys())

permutations_of_parameters = [
    dict(zip(keys, values))
    for values in product(*(sample_parameters[k] for k in keys))
]

policies = [
    cart_model.ActorPolicyContinuousSpace(**params)
    for params in permutations_of_parameters
]
best_policy = None
best_reward = float('-inf')

In [ ]:

context = cart_model.Cart_model(human=False)
episodes = 50000

for p in policies:
    total_reward = 0
    tqdm.write(f"Evaluating policy with parameters: {p.get_parameters()}")

    with tqdm(total=episodes, desc="Training", leave=True) as pbar:
        for i in range(episodes):
            startingState, _ = context.reset()
            episode = cart_model.Episode()
            p.new_episode(startingState)

            state, action, reward = episode.one_step(context, p)
            while True:
                p.update(reward, state, action, episode.terminated)
                if episode.terminated or episode.truncated:
                    break
                state, action, reward = episode.one_step(context, p)

            pbar.update(1)

    for i in range(100):
        ep = cart_model.Episode()
        ep.run(context, p)
        total_reward += ep.total_reward

    average_reward = total_reward / 100
    if average_reward > best_reward:
        best_reward = average_reward
        best_policy = p

In [ ]:
viewableCart=cart_model.Cart_model(human=False)

In [ ]:
max_reward=0
p=best_policy
print("Best policy parameters: ", p.get_parameters())
print("It got an average reward of: ", best_reward)
for i in range(100):
    slowEP=cart_model.Episode()
    slowEP.run(viewableCart, p)
    if slowEP.total_reward>max_reward:
        max_reward=slowEP.total_reward

print("Max reward: ", max_reward)


In [ ]:
viewableCart.close()